<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
// Интерфейс для идентифицируемых объектов
public interface IIdentifiable
{
    string GetId();
    string GetEntityType();
    DateTime GetCreatedDate();
    bool ValidateId();
}

// Интерфейс для сервиса уведомлений
public interface INotificationService
{
    void SendNotification(string message, Person recipient);
    void SendBulkNotification(string message, List<Person> recipients);
}

// Интерфейс для сервиса аналитики
public interface IAnalyticsService
{
    void TrackPersonCreation(Person person);
    void TrackInteraction(string interactionType, Person person1, Person person2);
    void GenerateReport();
}

// Реализация сервиса уведомлений
public class EmailNotificationService : INotificationService
{
    public void SendNotification(string message, Person recipient)
    {
        Console.WriteLine($"📧 Email отправлен на {recipient.Email}: {message}");
    }

    public void SendBulkNotification(string message, List<Person> recipients)
    {
        Console.WriteLine($"📧 Массовая рассылка на {recipients.Count} человек: {message}");
        foreach (var recipient in recipients.Take(3))
        {
            Console.WriteLine($"   - {recipient.Name} ({recipient.Email})");
        }
        if (recipients.Count > 3)
            Console.WriteLine($"   ... и еще {recipients.Count - 3} человек");
    }
}

// Реализация сервиса аналитики
public class AnalyticsService : IAnalyticsService
{
    private List<string> _events = new List<string>();

    public void TrackPersonCreation(Person person)
    {
        string eventInfo = $"[{DateTime.Now:HH:mm:ss}] Создан {person.GetType().Name}: {person.Name}";
        _events.Add(eventInfo);
        Console.WriteLine($"📊 Аналитика: {eventInfo}");
    }

    public void TrackInteraction(string interactionType, Person person1, Person person2)
    {
        string eventInfo = $"[{DateTime.Now:HH:mm:ss}] Взаимодействие '{interactionType}' между {person1.Name} и {person2.Name}";
        _events.Add(eventInfo);
        Console.WriteLine($"📊 Аналитика: {eventInfo}");
    }

    public void GenerateReport()
    {
        Console.WriteLine("\n=== ОТЧЕТ АНАЛИТИКИ ===");
        Console.WriteLine($"Всего событий: {_events.Count}");
        foreach (var eventItem in _events.TakeLast(10))
        {
            Console.WriteLine($"  • {eventItem}");
        }
    }
}

// Контейнер зависимостей
public class DependencyContainer
{
    private static Dictionary<Type, object> _services = new Dictionary<Type, object>();

    public static void Register<T>(T service)
    {
        _services[typeof(T)] = service;
    }

    public static T Resolve<T>()
    {
        if (_services.ContainsKey(typeof(T)))
        {
            return (T)_services[typeof(T)];
        }
        throw new InvalidOperationException($"Сервис {typeof(T).Name} не зарегистрирован");
    }
}

// Базовый класс Person с явной реализацией интерфейса IIdentifiable
public class Person : IIdentifiable
{
    public string Name { get; set; }
    public int Age { get; set; }
    public string Gender { get; set; }
    public string Email { get; set; }
    public string PhoneNumber { get; set; }
    public bool IsMarried { get; set; }
    public string Nationality { get; set; }
    public DateTime RegistrationDate { get; set; }
    
    // Новые атрибуты
    public string Address { get; set; }
    public string EmergencyContact { get; set; }
    public List<string> Languages { get; set; }
    public decimal MonthlyIncome { get; set; } // Общее свойство для всех типов доходов

    // Явная реализация интерфейса IIdentifiable
    string IIdentifiable.GetId()
    {
        return $"PERSON_{Name.Replace(" ", "_").ToUpper()}_{RegistrationDate:yyyyMMdd}";
    }

    string IIdentifiable.GetEntityType()
    {
        return "Person";
    }

    DateTime IIdentifiable.GetCreatedDate()
    {
        return RegistrationDate;
    }

    bool IIdentifiable.ValidateId()
    {
        var identifiable = (IIdentifiable)this;
        return !string.IsNullOrEmpty(identifiable.GetId()) && 
               identifiable.GetId().StartsWith("PERSON_");
    }

    public Person() 
    {
        RegistrationDate = DateTime.Now;
        Languages = new List<string>();
    }

    public Person(string name, int age, string gender, string email, string phoneNumber, 
                  bool isMarried, string nationality = "Не указана", string address = "", 
                  string emergencyContact = "", decimal monthlyIncome = 0)
    {
        Name = name;
        Age = age;
        Gender = gender;
        Email = email;
        PhoneNumber = phoneNumber;
        IsMarried = isMarried;
        Nationality = nationality;
        RegistrationDate = DateTime.Now;
        Address = address;
        EmergencyContact = emergencyContact;
        MonthlyIncome = monthlyIncome;
        Languages = new List<string>();
    }

    public virtual string GetInfo()
    {
        return $"Имя: {Name}, Возраст: {Age}, Пол: {Gender}, Email: {Email}, Телефон: {PhoneNumber}, " +
               $"В браке: {(IsMarried ? "Да" : "Нет")}, Национальность: {Nationality}, " +
               $"Адрес: {Address}, Экстренный контакт: {EmergencyContact}, Месячный доход: {MonthlyIncome:C}";
    }

    // Новые методы
    public virtual void AddLanguage(string language)
    {
        if (!Languages.Contains(language))
        {
            Languages.Add(language);
            Console.WriteLine($"{Name} выучил(а) язык: {language}");
        }
    }

    public virtual void RemoveLanguage(string language)
    {
        if (Languages.Remove(language))
        {
            Console.WriteLine($"{Name} забыл(а) язык: {language}");
        }
    }

    public string GetLanguages()
    {
        return Languages.Count > 0 ? string.Join(", ", Languages) : "Языки не указаны";
    }

    public virtual void RequestTimeOff(int days, string reason)
    {
        Console.WriteLine($"{Name} запрашивает отпуск на {days} дней. Причина: {reason}");
        
        try
        {
            var notificationService = DependencyContainer.Resolve<INotificationService>();
            notificationService.SendNotification($"Запрос отпуска от {Name} на {days} дней: {reason}", this);
        }
        catch (InvalidOperationException)
        {
            Console.WriteLine("Сервис уведомлений недоступен");
        }
    }

    public virtual void CalculateAnnualIncome()
    {
        decimal annualIncome = MonthlyIncome * 12;
        Console.WriteLine($"{Name}: Годовой доход составляет {annualIncome:C}");
    }
}

public class Student : Person, IIdentifiable
{
    public string University { get; set; }
    public int Course { get; set; }
    public List<string> Courses { get; set; }
    public string StudentId { get; set; }
    public double AverageGrade { get; set; }
    public string Faculty { get; set; }
    public bool HasScholarship { get; set; }
    
    // Новые атрибуты
    public string Major { get; set; }
    public int CreditsCompleted { get; set; }
    public DateTime ExpectedGraduation { get; set; }
    public bool IsInternational { get; set; }

    // Свойство для стипендии (дополнительный доход)
    public decimal ScholarshipAmount { get; set; }

    // Явная реализация интерфейса IIdentifiable
    string IIdentifiable.GetId()
    {
        return StudentId ?? $"STUDENT_{Name.Replace(" ", "_").ToUpper()}_{University?.Replace(" ", "_").ToUpper()}";
    }

    string IIdentifiable.GetEntityType()
    {
        return "Student";
    }

    DateTime IIdentifiable.GetCreatedDate()
    {
        return RegistrationDate;
    }

    bool IIdentifiable.ValidateId()
    {
        var identifiable = (IIdentifiable)this;
        return !string.IsNullOrEmpty(identifiable.GetId()) && 
               (identifiable.GetId().StartsWith("STUDENT_") || identifiable.GetId().StartsWith("S"));
    }

    public Student(string name, int age, string gender, string email, string phoneNumber, 
                   bool isMarried, string university, int course, string studentId, 
                   double averageGrade, string faculty = "Не указан", bool hasScholarship = false,
                   string major = "", int creditsCompleted = 0, bool isInternational = false,
                   decimal scholarshipAmount = 0)
        : base(name, age, gender, email, phoneNumber, isMarried)
    {
        University = university;
        Course = course;
        StudentId = studentId;
        AverageGrade = averageGrade;
        Faculty = faculty;
        HasScholarship = hasScholarship;
        Major = major;
        CreditsCompleted = creditsCompleted;
        IsInternational = isInternational;
        ScholarshipAmount = scholarshipAmount;
        MonthlyIncome = scholarshipAmount; // Стипендия как основной доход для студента
        ExpectedGraduation = DateTime.Now.AddYears(4 - course);
        Courses = new List<string>();
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Университет: {University}, Факультет: {Faculty}, " +
               $"Курс: {Course}, ID: {StudentId}, Средний балл: {AverageGrade:F1}, " +
               $"Стипендия: {(HasScholarship ? "Да" : "Нет")}, Специальность: {Major}, " +
               $"Завершено кредитов: {CreditsCompleted}, Международный: {(IsInternational ? "Да" : "Нет")}";
    }

    // Новые методы
    public void AddCourse(string course)
    {
        if (!Courses.Contains(course))
        {
            Courses.Add(course);
            CreditsCompleted += 3;
            Console.WriteLine($"{Name} добавлен в курс: {course}. Всего кредитов: {CreditsCompleted}");
        }
    }

    public void RemoveCourse(string course)
    {
        if (Courses.Remove(course))
        {
            CreditsCompleted = Math.Max(0, CreditsCompleted - 3);
            Console.WriteLine($"{Name} удален из курса: {course}. Всего кредитов: {CreditsCompleted}");
        }
    }

    public void CalculateGPA()
    {
        double gpa = AverageGrade / 5.0 * 4.0;
        Console.WriteLine($"{Name}: GPA составляет {gpa:F2} из 4.0");
    }

    public override void RequestTimeOff(int days, string reason)
    {
        Console.WriteLine($"{Name} (студент) запрашивает академический отпуск на {days} дней. Причина: {reason}");
        
        try
        {
            var notificationService = DependencyContainer.Resolve<INotificationService>();
            notificationService.SendNotification($"Академический отпуск от {Name} на {days} дней: {reason}", this);
        }
        catch (InvalidOperationException)
        {
            Console.WriteLine("Сервис уведомлений недоступен");
        }
    }

    public void ApplyForGraduation()
    {
        if (CreditsCompleted >= 120)
        {
            Console.WriteLine($"{Name} подает заявление на выпуск. Ожидаемая дата выпуска: {ExpectedGraduation:yyyy-MM-dd}");
        }
        else
        {
            Console.WriteLine($"{Name} не может выпуститься. Необходимо {120 - CreditsCompleted} дополнительных кредитов.");
        }
    }

    public override void CalculateAnnualIncome()
    {
        decimal annualIncome = MonthlyIncome * 9; // Стипендия только 9 месяцев в году
        Console.WriteLine($"{Name}: Годовой доход (стипендия) составляет {annualIncome:C}");
    }
}

public class Employee : Person, IIdentifiable
{
    public string Company { get; set; }
    public string Position { get; set; }
    public string EmployeeId { get; set; }
    public int VacationDays { get; set; }
    public string Department { get; set; }
    public int YearsInCompany { get; set; }
    
    // Новые атрибуты
    public string Manager { get; set; }
    public string WorkSchedule { get; set; }
    public bool IsRemote { get; set; }
    public int ProjectsCompleted { get; set; }

    // Явная реализация интерфейса IIdentifiable
    string IIdentifiable.GetId()
    {
        return EmployeeId ?? $"EMPLOYEE_{Name.Replace(" ", "_").ToUpper()}_{Company?.Replace(" ", "_").ToUpper()}";
    }

    string IIdentifiable.GetEntityType()
    {
        return "Employee";
    }

    DateTime IIdentifiable.GetCreatedDate()
    {
        return RegistrationDate;
    }

    bool IIdentifiable.ValidateId()
    {
        var identifiable = (IIdentifiable)this;
        return !string.IsNullOrEmpty(identifiable.GetId()) && 
               (identifiable.GetId().StartsWith("EMPLOYEE_") || identifiable.GetId().StartsWith("E"));
    }

    public Employee(string name, int age, string gender, string email, string phoneNumber, 
                    bool isMarried, string company, decimal salary, string employeeId, 
                    int vacationDays, string department = "Не указан", int yearsInCompany = 0, 
                    string position = "Сотрудник", string manager = "", string workSchedule = "9-18",
                    bool isRemote = false, int projectsCompleted = 0)
        : base(name, age, gender, email, phoneNumber, isMarried, monthlyIncome: salary)
    {
        Company = company;
        Position = position;
        EmployeeId = employeeId;
        VacationDays = vacationDays;
        Department = department;
        YearsInCompany = yearsInCompany;
        Manager = manager;
        WorkSchedule = workSchedule;
        IsRemote = isRemote;
        ProjectsCompleted = projectsCompleted;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Компания: {Company}, Департамент: {Department}, " +
               $"Должность: {Position}, ID: {EmployeeId}, " +
               $"Лет в компании: {YearsInCompany}, Отпускных дней: {VacationDays}, " +
               $"Менеджер: {Manager}, График: {WorkSchedule}, Удаленная работа: {(IsRemote ? "Да" : "Нет")}, " +
               $"Завершено проектов: {ProjectsCompleted}";
    }

    // Новые методы
    public void CompleteProject(string projectName)
    {
        ProjectsCompleted++;
        Console.WriteLine($"{Name} завершил(а) проект '{projectName}'. Всего завершено проектов: {ProjectsCompleted}");
        
        try
        {
            var analyticsService = DependencyContainer.Resolve<IAnalyticsService>();
            analyticsService.TrackInteraction("project_completion", this, this);
        }
        catch (InvalidOperationException) { }
    }

    public void RequestPromotion(string desiredPosition)
    {
        Console.WriteLine($"{Name} запрашивает повышение до должности: {desiredPosition}");
        
        try
        {
            var notificationService = DependencyContainer.Resolve<INotificationService>();
            notificationService.SendNotification($"Запрос повышения от {Name} на должность {desiredPosition}", this);
        }
        catch (InvalidOperationException)
        {
            Console.WriteLine("Сервис уведомлений недоступен");
        }
    }

    public override void CalculateAnnualIncome()
    {
        decimal annualSalary = MonthlyIncome * 12;
        decimal bonus = annualSalary * 0.1m * YearsInCompany;
        decimal total = annualSalary + bonus;
        
        Console.WriteLine($"{Name}: Годовая зарплата {annualSalary:C} + бонус {bonus:C} = {total:C}");
    }

    public void ScheduleMeeting(string withPerson, DateTime date, string topic)
    {
        Console.WriteLine($"{Name} назначает встречу с {withPerson} на {date:yyyy-MM-dd HH:mm}. Тема: {topic}");
    }

    public override void RequestTimeOff(int days, string reason)
    {
        Console.WriteLine($"{Name} (сотрудник) запрашивает отпуск на {days} дней. Причина: {reason}");
        
        if (days > VacationDays)
        {
            Console.WriteLine($"⚠️  Недостаточно отпускных дней. Доступно: {VacationDays}, Запрошено: {days}");
        }
        
        try
        {
            var notificationService = DependencyContainer.Resolve<INotificationService>();
            notificationService.SendNotification($"Запрос отпуска сотрудника {Name} на {days} дней: {reason}", this);
        }
        catch (InvalidOperationException)
        {
            Console.WriteLine("Сервис уведомлений недоступен");
        }
    }

    // Специфичный метод для сотрудника
    public void RequestSalaryIncrease(decimal amount)
    {
        Console.WriteLine($"{Name} запрашивает увеличение зарплаты на {amount:C}");
        MonthlyIncome += amount;
        Console.WriteLine($"Новая месячная зарплата: {MonthlyIncome:C}");
    }
}

public class Teacher : Person, IIdentifiable
{
    public string Subject { get; set; }
    public int Experience { get; set; }
    public List<Student> Students { get; set; }
    public string TeacherId { get; set; }
    public string OfficeNumber { get; set; }
    public string AcademicDegree { get; set; }
    public int PublicationsCount { get; set; }
    
    // Новые атрибуты
    public string ResearchArea { get; set; }
    public int OfficeHours { get; set; }
    public bool IsTenured { get; set; }
    public List<string> TeachingCertifications { get; set; }

    // Явная реализация интерфейса IIdentifiable
    string IIdentifiable.GetId()
    {
        return TeacherId ?? $"TEACHER_{Name.Replace(" ", "_").ToUpper()}_{Subject?.Replace(" ", "_").ToUpper()}";
    }

    string IIdentifiable.GetEntityType()
    {
        return "Teacher";
    }

    DateTime IIdentifiable.GetCreatedDate()
    {
        return RegistrationDate;
    }

    bool IIdentifiable.ValidateId()
    {
        var identifiable = (IIdentifiable)this;
        return !string.IsNullOrEmpty(identifiable.GetId()) && 
               (identifiable.GetId().StartsWith("TEACHER_") || identifiable.GetId().StartsWith("T"));
    }

    public Teacher(string name, int age, string gender, string email, string phoneNumber, 
                   bool isMarried, string subject, int experience, string teacherId, 
                   string officeNumber, string academicDegree = "Кандидат наук", 
                   int publicationsCount = 0, string researchArea = "", int officeHours = 10,
                   bool isTenured = false, decimal salary = 0)
        : base(name, age, gender, email, phoneNumber, isMarried, monthlyIncome: salary)
    {
        Subject = subject;
        Experience = experience;
        TeacherId = teacherId;
        OfficeNumber = officeNumber;
        AcademicDegree = academicDegree;
        PublicationsCount = publicationsCount;
        ResearchArea = researchArea;
        OfficeHours = officeHours;
        IsTenured = isTenured;
        TeachingCertifications = new List<string>();
        Students = new List<Student>();
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Предмет: {Subject}, Опыт: {Experience} лет, " +
               $"ID: {TeacherId}, Кабинет: {OfficeNumber}, Ученая степень: {AcademicDegree}, " +
               $"Публикации: {PublicationsCount}, Область исследований: {ResearchArea}, " +
               $"Часы приема: {OfficeHours}, Тенура: {(IsTenured ? "Да" : "Нет")}, " +
               $"Сертификаты: {TeachingCertifications.Count}";
    }

    // Новые методы
    public void AddCertification(string certification)
    {
        if (!TeachingCertifications.Contains(certification))
        {
            TeachingCertifications.Add(certification);
            Console.WriteLine($"{Name} получил(а) сертификат: {certification}");
        }
    }

    public void ScheduleOfficeHours(string schedule)
    {
        OfficeHours = 10;
        Console.WriteLine($"{Name} установил(а) часы приема: {schedule}");
    }

    public void MentorStudent(Student student)
    {
        if (!Students.Contains(student))
        {
            Students.Add(student);
            Console.WriteLine($"{Name} стал(а) ментором для студента {student.Name}");
            
            try
            {
                var analyticsService = DependencyContainer.Resolve<IAnalyticsService>();
                analyticsService.TrackInteraction("mentoring", this, student);
            }
            catch (InvalidOperationException) { }
        }
    }

    public override void RequestTimeOff(int days, string reason)
    {
        Console.WriteLine($"{Name} (преподаватель) запрашивает академический отпуск на {days} дней. Причина: {reason}");
        
        if (days > 30)
        {
            Console.WriteLine("⚠️  Длительный отпуск требует одобрения декана");
        }
        
        try
        {
            var notificationService = DependencyContainer.Resolve<INotificationService>();
            notificationService.SendNotification($"Академический отпуск преподавателя {Name} на {days} дней: {reason}", this);
        }
        catch (InvalidOperationException)
        {
            Console.WriteLine("Сервис уведомлений недоступен");
        }
    }

    public void SubmitResearchProposal(string title, string fundingAgency)
    {
        Console.WriteLine($"{Name} подает исследовательское предложение: '{title}' в {fundingAgency}");
        PublicationsCount++;
    }

    public override void CalculateAnnualIncome()
    {
        decimal baseAnnualIncome = MonthlyIncome * 12;
        decimal researchBonus = PublicationsCount * 5000; // Бонус за публикации
        decimal total = baseAnnualIncome + researchBonus;
        
        Console.WriteLine($"{Name}: Годовая зарплата {baseAnnualIncome:C} + бонус за публикации {researchBonus:C} = {total:C}");
    }
}

// Обновленный дженерик-класс с использованием dependency injection
public class PersonRegistry<T> where T : Person
{
    private List<T> _people = new List<T>();
    private INotificationService _notificationService;
    private IAnalyticsService _analyticsService;

    // Constructor injection
    public PersonRegistry(INotificationService notificationService, IAnalyticsService analyticsService)
    {
        _notificationService = notificationService;
        _analyticsService = analyticsService;
    }

    public void AddPerson(T person)
    {
        _people.Add(person);
        Console.WriteLine($"{person.Name} добавлен(а) в реестр.");
        
        _analyticsService.TrackPersonCreation(person);
        
        if (person is IIdentifiable identifiable)
        {
            Console.WriteLine($"ID объекта: {identifiable.GetId()}, Тип: {identifiable.GetEntityType()}");
        }
    }

    // Демонстрация работы с интерфейсом IIdentifiable
    public void ValidateAllIds()
    {
        Console.WriteLine("\n=== ПРОВЕРКА ИДЕНТИФИКАТОРОВ ===");
        foreach (var person in _people.OfType<IIdentifiable>())
        {
            bool isValid = person.ValidateId();
            string status = isValid ? "✓ ВАЛИДЕН" : "✗ НЕВАЛИДЕН";
            Console.WriteLine($"{person.GetEntityType()} {person.GetId()}: {status}");
        }
    }

    public void PrintAllIdentifiableInfo()
    {
        Console.WriteLine("\n=== ИНФОРМАЦИЯ ОБ ИДЕНТИФИЦИРУЕМЫХ ОБЪЕКТАХ ===");
        foreach (var identifiable in _people.OfType<IIdentifiable>())
        {
            Console.WriteLine($"{identifiable.GetEntityType()}: {identifiable.GetId()} " +
                            $"(создан: {identifiable.GetCreatedDate():yyyy-MM-dd})");
        }
    }

    public T FindPerson(string name)
    {
        return _people.FirstOrDefault(p => p.Name.Equals(name, StringComparison.OrdinalIgnoreCase));
    }

    public T FindPerson(string id, bool isIdSearch)
    {
        if (!isIdSearch) return FindPerson(id);
        
        foreach (var person in _people.OfType<IIdentifiable>())
        {
            if (person.GetId() == id)
                return (T)person;
        }
        return null;
    }

    public List<T> GetAllPeople()
    {
        return new List<T>(_people);
    }

    public List<T> GetPeopleByAge(int minAge, int maxAge)
    {
        return _people.Where(p => p.Age >= minAge && p.Age <= maxAge).ToList();
    }

    public void PrintRegistryStats()
    {
        Console.WriteLine($"\n=== СТАТИСТИКА РЕЕСТРА ===");
        Console.WriteLine($"Всего людей: {_people.Count}");
        Console.WriteLine($"Студентов: {_people.OfType<Student>().Count()}");
        Console.WriteLine($"Преподавателей: {_people.OfType<Teacher>().Count()}");
        Console.WriteLine($"Сотрудников: {_people.OfType<Employee>().Count()}");
        Console.WriteLine($"Средний возраст: {_people.Average(p => p.Age):F1} лет");
        Console.WriteLine($"Средний месячный доход: {_people.Average(p => p.MonthlyIncome):C}");
        
        var identifiableCount = _people.OfType<IIdentifiable>().Count();
        Console.WriteLine($"Идентифицируемых объектов: {identifiableCount}/{_people.Count}");
    }

    public void SendBulkNotificationToStudents(string message)
    {
        var students = _people.OfType<Student>().Cast<Person>().ToList();
        _notificationService.SendBulkNotification(message, students);
    }
}

// Основной код программы (глобальный код)
Console.WriteLine("=== РАСШИРЕННАЯ СИСТЕМА С DI И ИНТЕРФЕЙСАМИ ===\n");

// Регистрация зависимостей
DependencyContainer.Register<INotificationService>(new EmailNotificationService());
DependencyContainer.Register<IAnalyticsService>(new AnalyticsService());

// Создание реестра с внедренными зависимостями
var notificationService = DependencyContainer.Resolve<INotificationService>();
var analyticsService = DependencyContainer.Resolve<IAnalyticsService>();

PersonRegistry<Person> registry = new PersonRegistry<Person>(notificationService, analyticsService);

// Создание объектов с новыми атрибутами
Student student1 = new Student("Артур", 19, "Мужской", "alshin06@gmail.com", "+79526759898", 
    false, "ТИУ", 2, "S2024001", 4.2, "Информационные технологии", true,
    "Компьютерные науки", 45, true, 5000);

Student student2 = new Student("Анна", 20, "Женский", "anna@university.ru", "+79098765432", 
    false, "ТИУ", 3, "S2023005", 4.8, "Компьютерные науки", false,
    "Искусственный интеллект", 75, false, 7000);

Employee employee1 = new Employee("Иван", 35, "Мужской", "ivan@sber.ru", "+79501234567", 
    true, "Сбербанк", 75000, "E1001", 21, "ИТ-отдел", 5, "Менеджер проектов",
    "Петр Сидоров", "9-18", false, 12);

Employee employee2 = new Employee("Елена", 28, "Женский", "elena@sber.ru", "+79507654321", 
    false, "Сбербанк", 65000, "E1002", 18, "Аналитика", 2, "Старший аналитик",
    "Иван Петров", "10-19", true, 8);

Teacher teacher1 = new Teacher("Мария", 45, "Женский", "maria@university.ru", "+79215555555", 
    true, "История", 13, "T050", "101A", "Доктор наук", 15,
    "Средневековая история", 8, true, 90000);

Teacher teacher2 = new Teacher("Сергей", 50, "Мужской", "sergey@university.ru", "+79216666666", 
    true, "Информатика", 20, "T025", "205B", "Профессор", 25,
    "Машинное обучение", 6, true, 120000);

// Добавление в реестр
registry.AddPerson(student1);
registry.AddPerson(student2);
registry.AddPerson(employee1);
registry.AddPerson(employee2);
registry.AddPerson(teacher1);
registry.AddPerson(teacher2);

// Демонстрация работы с интерфейсом IIdentifiable
registry.ValidateAllIds();
registry.PrintAllIdentifiableInfo();

// Демонстрация новых методов
Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ НОВЫХ МЕТОДОВ ===");

student1.AddLanguage("Английский");
student1.AddLanguage("Немецкий");
student1.AddCourse("Машинное обучение");
student1.CalculateGPA();
student1.ApplyForGraduation();

employee1.CompleteProject("Автоматизация отчетности");
employee1.RequestPromotion("Ведущий менеджер проектов");
employee1.CalculateAnnualIncome();
employee1.ScheduleMeeting("Елена", DateTime.Now.AddDays(1), "Обсуждение нового проекта");

teacher1.AddCertification("Преподаватель высшей школы");
teacher1.MentorStudent(student1);
teacher1.SubmitResearchProposal("Новые методы анализа исторических данных", "РНФ");
teacher1.ScheduleOfficeHours("Пн, Ср 14:00-16:00");

// Демонстрация запросов отпуска с DI
Console.WriteLine("\n=== ЗАПРОСЫ ОТПУСКОВ ===");
student1.RequestTimeOff(10, "Медицинские причины");
employee1.RequestTimeOff(14, "Ежегодный отпуск");
teacher1.RequestTimeOff(30, "Научная работа");

// Массовая рассылка через DI
Console.WriteLine("\n=== МАССОВАЯ РАССЫЛКА ===");
registry.SendBulkNotificationToStudents("Напоминание о начале сессии!");

// Статистика и отчеты
registry.PrintRegistryStats();
analyticsService.GenerateReport();

// Демонстрация полиморфизма с интерфейсами
Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ ПОЛИМОРФИЗМА С ИНТЕРФЕЙСАМИ ===");
List<IIdentifiable> identifiables = new List<IIdentifiable> { student1, employee1, teacher1 };

foreach (var identifiable in identifiables)
{
    Console.WriteLine($"Тип: {identifiable.GetEntityType()}, ID: {identifiable.GetId()}, " +
                    $"Создан: {identifiable.GetCreatedDate():yyyy-MM-dd}, " +
                    $"Валиден: {identifiable.ValidateId()}");
}

// Демонстрация работы с доходами
Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ РАСЧЕТА ДОХОДОВ ===");
student1.CalculateAnnualIncome();
employee1.CalculateAnnualIncome();
teacher1.CalculateAnnualIncome();

// Дополнительные демонстрации
Console.WriteLine("\n=== ДОПОЛНИТЕЛЬНЫЕ ВОЗМОЖНОСТИ ===");
employee1.RequestSalaryIncrease(10000);
teacher1.CalculateAnnualIncome();

=== РАСШИРЕННАЯ СИСТЕМА С DI И ИНТЕРФЕЙСАМИ ===

Артур добавлен(а) в реестр.
📊 Аналитика: [03:14:19] Создан Student: Артур
ID объекта: S2024001, Тип: Student
Анна добавлен(а) в реестр.
📊 Аналитика: [03:14:19] Создан Student: Анна
ID объекта: S2023005, Тип: Student
Иван добавлен(а) в реестр.
📊 Аналитика: [03:14:19] Создан Employee: Иван
ID объекта: E1001, Тип: Employee
Елена добавлен(а) в реестр.
📊 Аналитика: [03:14:19] Создан Employee: Елена
ID объекта: E1002, Тип: Employee
Мария добавлен(а) в реестр.
📊 Аналитика: [03:14:19] Создан Teacher: Мария
ID объекта: T050, Тип: Teacher
Сергей добавлен(а) в реестр.
📊 Аналитика: [03:14:19] Создан Teacher: Сергей
ID объекта: T025, Тип: Teacher

=== ПРОВЕРКА ИДЕНТИФИКАТОРОВ ===
Student S2024001: ✓ ВАЛИДЕН
Student S2023005: ✓ ВАЛИДЕН
Employee E1001: ✓ ВАЛИДЕН
Employee E1002: ✓ ВАЛИДЕН
Teacher T050: ✓ ВАЛИДЕН
Teacher T025: ✓ ВАЛИДЕН

=== ИНФОРМАЦИЯ ОБ ИДЕНТИФИЦИРУЕМЫХ ОБЪЕКТАХ ===
Student: S2024001 (создан: 2025-11-08)
Student: S2023005 (создан: 2025